# AMIGOS Full Benchmarks — CIB-Net / HCAN-CIB Replication Suite

Replicates the complete **DEAP & DREAMER** evaluation protocol on the **AMIGOS** multimodal dataset (`Data_Preprocessed_P01.mat` ... `Data_Preprocessed_P40.mat`):
- **Phase 1:** DE (56-D EEG) + Peripheral (18-D ECG/GSR) feature extraction, Domain-Shift Diagnostics (`D0`-`D4` on `eeg`, `periph`, `concat`), and Shallow Baselines (`T0, LR_*, SVC_*, LR_late` under `none` & `zscore`).
- **Phase 2A:** Deep EEG-Only Baselines (`PlainCNN`, `EEGNet`, `ACRNN` under `none` + fair `EEGNet` under `zscore`).
- **Phase 2B:** Multimodal `HCAN-CIB` suite (`base`, `zscore` [transductive], `v_smote`, `gamo`, `deepsmote`, plus optional `class_weight`, `logit_adj`, and `no_gating` ablations).
- **Phase 3:** Paired Wilcoxon + Holm statistical comparison (`AMIGOS_Comprehensive_Results_{target}.md`), 4-region Stage B Gate Heatmaps, ROC/PR curves, t-SNE embeddings, and one-click output bundle zipping.

**Kaggle Input Setup:** Attach `nsut123/amigos` (40 subjects, recommended) or `nsut123/amigos-dataset` (15 subjects) + upload `CIB-Net-AMIGOS` code dataset (or copy into `/kaggle/working/CIB-Net-AMIGOS`). Select **GPU T4 x2** or **P100** accelerator.

In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path
import torch

os.environ["PYTHONUNBUFFERED"] = "1"

print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available(), flush=True)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), flush=True)
if shutil.which("nvidia-smi"):
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"],
        check=False,
    )

# Auto-locate CIB-Net-AMIGOS codebase (supports GitHub clone, /kaggle/input mount, or local)
def locate_code_dir():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/kaggle/working/CIB-Net-AMIGOS"),
        Path("/kaggle/working/eeg_hcan_cib/CIB-Net-AMIGOS"),
    ]
    for search_root in (Path("/kaggle/working"), Path("/kaggle/input")):
        if search_root.exists():
            for p in sorted(search_root.rglob("run_all_amigos_kaggle.py")):
                candidates.append(p.parent)
    for c in candidates:
        if (c / "run_all_amigos_kaggle.py").exists():
            return c
    raise FileNotFoundError(
        "Could not locate 'run_all_amigos_kaggle.py'. Clone https://github.com/Suraj31shah/eeg_hcan_cib.git or attach CIB-Net-AMIGOS dataset."
    )

SRC_DIR = locate_code_dir()
WORK_CODE_DIR = Path("/kaggle/working/CIB-Net-AMIGOS") if Path("/kaggle/working").exists() else SRC_DIR

# Copy from read-only /kaggle/input into writable /kaggle/working if needed
if str(SRC_DIR).startswith("/kaggle/input") and SRC_DIR != WORK_CODE_DIR:
    if WORK_CODE_DIR.exists():
        shutil.rmtree(WORK_CODE_DIR)
    shutil.copytree(SRC_DIR, WORK_CODE_DIR)
    print(f"Copied read-only code from {SRC_DIR} -> {WORK_CODE_DIR}")

os.chdir(WORK_CODE_DIR)
if str(WORK_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_CODE_DIR))
print("Active CODE_DIR:", os.getcwd())

In [ ]:
# Install only lightweight missing packages without disturbing Kaggle's PyTorch build
import importlib.util

for mod_name, pip_name in [("yaml", "pyyaml"), ("seaborn", "seaborn"), ("h5py", "h5py"), ("tqdm", "tqdm")]:
    if importlib.util.find_spec(mod_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

# ===================== EXPERIMENT CONFIGURATION =====================
TARGET = "valence"            # Primary target: 'valence' | 'arousal' | 'dominance'
RUN_AROUSAL_TOO = False       # Set True to run arousal after valence finishes (recommended in Session 2 if time-limited)
BINARIZE = "fixed"            # 'fixed' (>5.0 on 1-9 scale) | 'median'
VIDEO_SUBSET = "short"        # 'short' (16 short clips <=240s) | 'all'
SEED = 0
ALSO_ZSCORE_ALL = False       # Set True to also run zscore variants of PlainCNN/ACRNN + V-SMOTE/GAMO/DeepSMOTE
INCLUDE_ABLATIONS = True      # Adds class_weight, logit_adj, and no_gating ablation
MAX_FOLDS = None              # Keep None for full LOSO; set small int (e.g. 2) for a quick smoke check
EPOCHS = None                 # Keep None for config default (40); set int to override
RESUME = True                 # Safely skips completed variants if Kaggle session restarts

RESULTS_ROOT = "/kaggle/working/amigos_results" if os.path.exists("/kaggle/working") else "results"
os.makedirs(RESULTS_ROOT, exist_ok=True)
print("RESULTS_ROOT:", RESULTS_ROOT)

In [ ]:
# Dataset Auto-Discovery & Quick Loader Sanity Check
from data.amigos_loader import find_amigos_files, WindowedAMIGOSDataset

files = find_amigos_files()
print(f"Discovered {len(files)} AMIGOS subject .mat files:")
print([f"P{sid:02d}" for sid, _ in files])
if not files:
    raise RuntimeError(
        "No Data_Preprocessed_P*.mat found under /kaggle/input! Attach nsut123/amigos or nsut123/amigos-dataset."
    )

DATA_DIR = os.path.commonpath([str(Path(p).parent) for _, p in files])
print("Inferred DATA_DIR:", DATA_DIR)

# Verify label ordering (col 0 = arousal, col 1 = valence) & signal integrity on first 2 subjects
ds_check = WindowedAMIGOSDataset(
    data_path=DATA_DIR,
    target=TARGET,
    binarize=BINARIZE,
    video_subset=VIDEO_SUBSET,
    normalize="none",
    subjects=[files[0][0], files[min(1, len(files) - 1)][0]],
)
x_e, x_p, y0, s0, t0 = ds_check[0]
print("Sample tensors -> EEG:", tuple(x_e.shape), "| Periph:", tuple(x_p.shape), "| label:", int(y0), "| subj:", int(s0))
assert torch.isfinite(x_e).all() and torch.isfinite(x_p).all(), "Non-finite values detected after preprocessing!"
del ds_check

## Phase 1: DE + Peripheral Extraction, Domain-Shift Diagnostics (`D0`-`D4`), & Shallow Baselines

In [ ]:
import glob
from IPython.display import display, Image, Markdown

def run_phase(phase: str, target: str):
    cmd = [
        sys.executable, "-u", "run_all_amigos_kaggle.py",
        "--phase", phase,
        "--target", target,
        "--binarize", BINARIZE,
        "--video_subset", VIDEO_SUBSET,
        "--results_root", RESULTS_ROOT,
        "--seed", str(SEED),
    ]
    if DATA_DIR:
        cmd += ["--data_path", DATA_DIR]
    if ALSO_ZSCORE_ALL:
        cmd.append("--also_zscore_all")
    if INCLUDE_ABLATIONS:
        cmd.append("--include_ablations")
    if MAX_FOLDS is not None:
        cmd += ["--max_folds", str(MAX_FOLDS)]
    if EPOCHS is not None:
        cmd += ["--epochs", str(EPOCHS)]
    if RESUME:
        cmd.append("--resume")
    print("Executing:", " ".join(cmd), flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    subprocess.run(cmd, env=env, check=True)

run_phase("stage1", TARGET)

# Display Diagnostics D1-D3 plots for eeg, periph, and concat
for fblock in ("eeg", "periph", "concat"):
    p = os.path.join(RESULTS_ROOT, f"diag_{TARGET}_{fblock}", "diagnostics.png")
    if os.path.exists(p):
        print(f"--- Diagnostics ({TARGET} / {fblock}) ---")
        display(Image(filename=p))

## Phase 2A: Deep EEG-Only Baselines (`PlainCNN`, `EEGNet`, `ACRNN` + `EEGNet Z-score`)

In [ ]:
run_phase("baselines", TARGET)

## Phase 2B: Multimodal `HCAN-CIB` Variants (`base`, `zscore`, `V-SMOTE`, `GAMO`, `DeepSMOTE` + Ablations)

In [ ]:
run_phase("hcan", TARGET)

## Optional Second Target (`arousal`) — Controlled via `RUN_AROUSAL_TOO`

In [ ]:
if RUN_AROUSAL_TOO and TARGET != "arousal":
    run_phase("all", "arousal")
else:
    print("Skipped secondary arousal run (set RUN_AROUSAL_TOO = True to enable).")

## Phase 3: Summary Report, Diagnostic Visualizations & Kaggle Output Zip

In [ ]:
import zipfile

targets_to_show = [TARGET] + (["arousal"] if RUN_AROUSAL_TOO and TARGET != "arousal" else [])

for tgt in targets_to_show:
    run_phase("summary", tgt)
    rep = os.path.join(RESULTS_ROOT, f"AMIGOS_Comprehensive_Results_{tgt}.md")
    if os.path.exists(rep):
        with open(rep, encoding="utf-8") as f:
            display(Markdown(f.read()))

    for folder in sorted(glob.glob(os.path.join(RESULTS_ROOT, f"hcan_*_{tgt}*"))):
        name = os.path.basename(folder)
        for img_name in ("gate_heatmap.png", "roc_pr_curves.png", "tsne_brain_embeddings.png"):
            ip = os.path.join(folder, "plots", img_name)
            if os.path.exists(ip):
                print(f"[{name}] {img_name}")
                display(Image(filename=ip))

# Bundle JSONs, NPZs, PNGs, and Markdown reports for clean Kaggle download
INCLUDE_WEIGHTS_IN_ZIP = False  # Keep False for lightweight <25 MB output archive
zip_out = "/kaggle/working/amigos_benchmark_outputs.zip" if os.path.exists("/kaggle/working") else os.path.join(RESULTS_ROOT, "amigos_benchmark_outputs.zip")
n_packed = 0
with zipfile.ZipFile(zip_out, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for root, _, fnames in os.walk(RESULTS_ROOT):
        for fn in fnames:
            if not INCLUDE_WEIGHTS_IN_ZIP and fn.endswith(".pth"):
                continue
            fp = os.path.join(root, fn)
            if os.path.abspath(fp) == os.path.abspath(zip_out):
                continue
            zf.write(fp, arcname=os.path.relpath(fp, RESULTS_ROOT))
            n_packed += 1
print(f"Created {zip_out} ({n_packed} files, {os.path.getsize(zip_out)/1e6:.2f} MB)")